# Entraînement YOLO compost sur Colab (GPU)

**Règle d'or : le code se modifie dans le repo et se commit, JAMAIS dans ce notebook.**
Ce notebook ne fait qu'orchestrer : clone, install, données, scripts, sauvegarde.
Colab est en LECTURE SEULE vis-à-vis de git : on clone, aucune cellule ne
commit ni ne push — rien de ce qui se passe ici n'apparaît sur GitHub.

Prérequis :
- runtime GPU (Exécution > Modifier le type d'exécution > T4 GPU) ;
- un token GitHub personnel classique (scope `repo`) dans les Secrets Colab
  sous `GITHUB_TOKEN` — il ne sert qu'au clone ;
- un zip par dataset sur Drive, nommés `MyDrive/compost/dataset_raw_<nom>.zip`
  (ex. `dataset_raw_zerowaste.zip`, `dataset_raw_taco.zip`) — chacun contient
  `images/` + `labels/` + `groups.csv` (sortie de `import_dataset.py`).
  La liste des `<nom>` se renseigne dans la variable `DATASETS` de la cellule 5.

In [ ]:
# 1. Clone du repo (token lu depuis les Secrets Colab — utilisé uniquement pour cloner)
BRANCH = 'yolo'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!rm -rf /content/repo   # on se place d'abord HORS du dossier pour pouvoir le supprimer sans erreur
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
# le code d'entraînement est le sous-dossier compost-yolo du repo
%cd /content/repo/compost-yolo

In [ ]:
# 2. Installation des dépendances
!pip install -q -e .

In [ ]:
# 3. (désactivée) Montage de Google Drive — désormais fait dans la cellule 5
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 4. (désactivée) Copie/dézippage de ZeroWaste — désormais géré par la liste DATASETS (cellule 5)
# !cp /content/drive/MyDrive/compost/dataset_raw.zip /content/
# !unzip -q -o /content/dataset_raw.zip -d /content/dataset_raw

In [ ]:
# 5. Données : datasets d'ENTRAÎNEMENT (externes) + test RÉEL compost (captures, HORS entraînement)
# Les 2 sessions de captures tombent toutes dans 'train' au split -> impossible de les tester
# proprement si on les entraîne. On les garde donc HORS entraînement = test compost 100% propre
# (revient à un pré-entraînement sur externes + mesure directe du domain gap).
TRAIN_DATASETS = ['zerowaste', 'taco', 'proj3', 'warp']   # -> /content/dataset (entraînement)
COMPOST_TEST   = 'captures'                                # -> test réel, JAMAIS entraîné

import os, shutil
from google.colab import drive
# Montage robuste : si /content/drive est un résidu (dossier NON monté laissé par un recyclage
# de VM), on le vide d'abord (sinon « Mountpoint must not already contain files »). Le garde-fou
# not ismount() garantit qu'on ne touche jamais à un vrai montage Drive.
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

# ENTRAÎNEMENT : copie + dézip + PREPARE (split) dans /content/dataset
for name in TRAIN_DATASETS:
    zip_path = f'/content/drive/MyDrive/compost/dataset_raw_{name}.zip'
    if not os.path.exists(zip_path):
        print(f'⚠️  {name} : {zip_path} INTROUVABLE sur Drive — ignoré (à uploader)')
        continue
    !cp {zip_path} /content/
    !unzip -q -o /content/dataset_raw_{name}.zip -d /content/dataset_raw_{name}
    !python scripts/prepare_dataset.py --source /content/dataset_raw_{name} --output /content/dataset

# TEST COMPOST : copie + dézip SEULEMENT (pas de prepare -> tout reste en test, voir cellule 6c)
ct_zip = f'/content/drive/MyDrive/compost/dataset_raw_{COMPOST_TEST}.zip'
if os.path.exists(ct_zip):
    !cp {ct_zip} /content/
    !unzip -q -o /content/dataset_raw_{COMPOST_TEST}.zip -d /content/dataset_raw_{COMPOST_TEST}
else:
    print(f'⚠️  {COMPOST_TEST} : {ct_zip} INTROUVABLE — éval compost (6c) impossible')

In [ ]:
# 5b. Histogramme par dataset INDIVIDUEL (brut) — externes (entraînement) + captures (test)
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
x = range(len(names))

for name in TRAIN_DATASETS + [COMPOST_TEST]:
    counts = Counter()
    for label_file in Path(f'/content/dataset_raw_{name}/labels').glob('*.txt'):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    if not counts:
        print(f"⚠️  {name} : aucun label trouvé (zip manquant sur Drive ?) — sauté")
        continue
    fig, ax = plt.subplots(figsize=(10, 3))
    bars = ax.bar(list(x), [counts[j] for j in x])
    ax.bar_label(bars, fmt='%d', padding=2)   # nombre exact sur chaque barre
    ax.set_yscale('log')                      # sans le log, les classes rares sont invisibles
    ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=20)
    ax.set_ylabel('instances (log)')
    ax.set_title(f"Dataset « {name} » — instances par classe (brut)")
    ax.grid(axis='y', alpha=0.3); fig.tight_layout(); plt.show()
    print(f"{name}: {sum(counts.values())} instances —",
          ", ".join(f"{names[j]}: {counts[j]}" for j in x))

In [ ]:
# 5c. Par GROUPE : datasets externes (entraînement) vs captures réelles (test compost)
#     (instances/classe + nombre de boîtes par image + résumé)
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
x = range(len(names))
GROUPES = {
    'externes (entraînement)':   TRAIN_DATASETS,
    'captures (test compost)':   [COMPOST_TEST],
}

def stats_groupe(noms):
    per_class, boxes_per_img, n_img, n_neg = Counter(), [], 0, 0
    for name in noms:
        idir = Path(f'/content/dataset_raw_{name}/images')
        ldir = Path(f'/content/dataset_raw_{name}/labels')
        for img in idir.glob('*'):
            if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            n_img += 1
            lf = ldir / f'{img.stem}.txt'
            lines = [l for l in lf.read_text().splitlines() if l.strip()] if lf.exists() else []
            boxes_per_img.append(len(lines))
            if not lines:
                n_neg += 1
            for l in lines:
                per_class[int(l.split()[0])] += 1
    return per_class, boxes_per_img, n_img, n_neg

for titre, noms in GROUPES.items():
    if not noms:
        continue
    pc, bpi, n_img, n_neg = stats_groupe(noms)
    if n_img == 0:
        print(f"⚠️  {titre} : aucune image trouvée pour {noms} — zip(s) manquant(s) sur Drive ?")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
    # gauche : instances par classe
    b = axes[0].bar(list(x), [pc[j] for j in x])
    axes[0].bar_label(b, fmt='%d', padding=2)
    if any(pc.values()):
        axes[0].set_yscale('log')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(names, rotation=20); axes[0].set_ylabel('instances (log)')
    axes[0].set_title(f'{titre} — instances par classe'); axes[0].grid(axis='y', alpha=0.3)
    # droite : nombre de boîtes par image (0 = image négative)
    cap = 10
    dist = Counter(min(k, cap) for k in bpi)
    xs = list(range(cap + 1))
    b2 = axes[1].bar(xs, [dist[k] for k in xs])
    axes[1].bar_label(b2, fmt='%d', padding=2)
    axes[1].set_xticks(xs); axes[1].set_xticklabels([str(k) for k in xs[:-1]] + [f'{cap}+'])
    axes[1].set_xlabel('boîtes par image'); axes[1].set_ylabel("nombre d'images")
    axes[1].set_title(f'{titre} — boîtes par image'); axes[1].grid(axis='y', alpha=0.3)
    fig.tight_layout(); plt.show()
    moy = sum(bpi) / len(bpi) if bpi else 0
    print(f"{titre}: {n_img} images dont {n_neg} négatives | {sum(pc.values())} instances | "
          f"{moy:.2f} boîtes/image en moyenne")
    print("   par classe :", ", ".join(f"{names[j]}: {pc[j]}" for j in x))

In [ ]:
# 5d. Datasets d'ENTRAÎNEMENT (externes, fusionnés) — par classe et par split
# (les captures ne sont PAS ici : elles servent de test compost, voir 5c / 6c)
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
splits = ['train', 'val', 'test']
counts = {s: Counter() for s in splits}
for s in splits:
    for lf in Path(f'/content/dataset/labels/{s}').glob('*.txt'):
        for line in lf.read_text().splitlines():
            if line.strip():
                counts[s][int(line.split()[0])] += 1

x = range(len(names)); width = 0.27
fig, ax = plt.subplots(figsize=(12, 4))
for i, s in enumerate(splits):
    b = ax.bar([v + (i - 1) * width for v in x], [counts[s][j] for j in x], width, label=s)
    ax.bar_label(b, fmt='%d', padding=2, fontsize=7)   # nombre exact sur chaque barre
if any(any(c.values()) for c in counts.values()):
    ax.set_yscale('log')
ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=20)
ax.set_ylabel('instances (log)')
ax.set_title("Entraînement (externes) — instances par classe et par split")
ax.legend(); ax.grid(axis='y', alpha=0.3); fig.tight_layout(); plt.show()
for s in splits:
    print(f"{s}: {sum(counts[s].values())} instances —",
          ", ".join(f"{names[j]}: {counts[s][j]}" for j in x))

In [ ]:
# 5e. Split STRATIFIÉ des captures : les 2 sessions dans TRAIN ET TEST (mêmes proportions)
#  - 20% de CHAQUE session -> TEST compost held-out ; 80% -> FINE-TUNING.
#  ⚠️ Fuite possible si des photos consécutives sont quasi identiques (même scène) : une en
#     train, une en test -> score test un peu optimiste. Atténué car les dispositions varient ;
#     vrai remède = plus de sessions distinctes.
import re, random, shutil
from pathlib import Path
from collections import Counter
import yaml

random.seed(42)
TEST_FRACTION = 0.2                 # part de CHAQUE session mise en test

src = Path(f'/content/dataset_raw_{COMPOST_TEST}')
def session_of(stem):               # heure dans WIN_AAAAMMJJ_HH_MM_SS_Pro
    m = re.search(r'_(\d{2})_\d{2}_\d{2}', stem)
    h = int(m.group(1)) if m else 0
    return 's1.5' if h >= 13 else 's1'

by_session = {}
for img in (src / 'images').glob('*'):
    if img.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        by_session.setdefault(session_of(img.stem), []).append(img)

assign = {}                          # image -> 'test' / 'finetune'
for sess, imgs in by_session.items():
    random.shuffle(imgs)
    n_test = round(len(imgs) * TEST_FRACTION)
    for i, img in enumerate(imgs):
        assign[img] = 'test' if i < n_test else 'finetune'

for sub in ('finetune', 'test'):
    for d in ('images', 'labels'):
        Path(f'/content/captures_{sub}/{d}').mkdir(parents=True, exist_ok=True)
for img, sub in assign.items():
    shutil.copy(img, f'/content/captures_{sub}/images/{img.name}')
    lf = src / 'labels' / f'{img.stem}.txt'
    if lf.exists():
        shutil.copy(lf, f'/content/captures_{sub}/labels/{lf.name}')

# data.yaml du TEST compost (tout en test, pas de re-split)
names = yaml.safe_load(open('configs/data.yaml'))['names']
Path('/content/captures_test/data.yaml').write_text(yaml.safe_dump(
    {'path': '/content/captures_test', 'train': 'images', 'val': 'images', 'test': 'images',
     'names': dict(enumerate(names))}, allow_unicode=True, sort_keys=False))

# FINE-TUNING : on prépare (split train/val interne) le set fine-tune
!python scripts/prepare_dataset.py --source /content/captures_finetune --output /content/dataset_finetune

# récap par session
recap = Counter((session_of(i.stem), s) for i, s in assign.items())
for sub in ('finetune', 'test'):
    n = len(list(Path(f'/content/captures_{sub}/images').glob('*')))
    detail = ", ".join(f"{sess}:{recap[(sess, sub)]}" for sess in sorted(by_session))
    print(f"captures {sub}: {n} images ({detail})")

In [ ]:
# 6. Entraînement (checkpoints sauvegardés sur Drive toutes les 10 epochs)
# Reprise après coupure : ajouter --resume /content/runs/train_xxx/weights/last.pt
!python scripts/train.py --data /content/dataset/data.yaml \
    --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 6b. ÉVAL A — test des datasets EXTERNES (perf générale, domaine centre de tri, PAS le compost)
# Cherche le best.pt du dernier entraînement EN LOCAL puis SUR DRIVE (marche même après
# expiration de session : /content est effacé, Drive non).
from pathlib import Path

search_dirs = [
    '/content/runs',
    '/content/drive/MyDrive/compost/runs',
    '/content/drive/MyDrive/compost/backups',
]
candidates = []
for d in search_dirs:
    candidates += Path(d).glob('train_*/weights/best.pt')
candidates = sorted(candidates, key=lambda p: p.stat().st_mtime)
assert candidates, ("Aucun best.pt trouvé (ni local ni Drive). "
                    "Drive est-il monté (cellule 5) et l'entraînement a-t-il tourné ?")
best = candidates[-1]
print('Modèle évalué (test externes) :', best)

!python scripts/evaluate.py \
    --weights {best} \
    --data /content/dataset/data.yaml \
    --split test \
    --runs-dir /content/runs

from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    for img in sorted(evals[-1].rglob('*confusion*.png')):
        print(img.name)
        display(Image(str(img)))

In [ ]:
# 6c. ÉVAL B — COMPOST (test RÉEL, AVANT fine-tuning) : sur le test held-out (20% de chaque session).
#     Mesure le domain gap du modèle pré-entraîné. À COMPARER avec l'ÉVAL C (après fine-tuning, 6e).
from pathlib import Path

# Pour RÉUTILISER un pré-entraînement précis (ex. relancer plusieurs fine-tunings depuis le MÊME
# pré-entraîné, sans refaire les ~5 h), colle son chemin ici. Sinon (None) = dernier train_* trouvé.
# ⚠️ Après un 1er fine-tuning, "dernier train_*" devient le modèle FINE-TUNÉ : pour repartir du
#    pré-entraînement, PIN son chemin ici (visible dans la sortie du 1er run / sur Drive).
PRETRAIN_PATH = None   # ex : '/content/drive/MyDrive/compost/runs/train_29-06_14h15/weights/best.pt'

if PRETRAIN_PATH:
    PRETRAIN = Path(PRETRAIN_PATH)
else:
    cands = []
    for d in ['/content/runs', '/content/drive/MyDrive/compost/runs',
              '/content/drive/MyDrive/compost/backups']:
        cands += Path(d).glob('train_*/weights/best.pt')
    assert cands, "Aucun best.pt trouvé — l'entraînement a-t-il tourné ?"
    PRETRAIN = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]
assert PRETRAIN.exists(), f"best.pt introuvable : {PRETRAIN}"
print('Modèle PRÉ-ENTRAÎNÉ évalué sur le COMPOST :', PRETRAIN)

!python scripts/evaluate.py --weights {PRETRAIN} --data /content/captures_test/data.yaml --split test --runs-dir /content/runs

from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    print('--- confusion COMPOST (AVANT fine-tuning) ---')
    for img in sorted(evals[-1].rglob('*confusion*.png')):
        print(img.name); display(Image(str(img)))

In [ ]:
# 6d. FINE-TUNING : repart du modèle PRÉ-ENTRAÎNÉ (PRETRAIN, défini en 6c) et continue sur le
#     compost (80% des captures, les 2 sessions), learning rate BAS (départ doux pour ne pas tout casser).
#  ⚠️ Test du mécanisme : peu d'images -> risque d'oubli/surapprentissage. Le vrai fine-tuning
#     robuste viendra avec PLUS de sessions (et éventuellement un mélange d'externes).
!python scripts/train.py --model {PRETRAIN} --data /content/dataset_finetune/data.yaml \
    --epochs 30 --lr0 0.001 \
    --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 6e. ÉVAL C — COMPOST APRÈS fine-tuning, sur le MÊME test held-out que la 6c.
#     Compare directement à l'ÉVAL B : le fine-tuning a-t-il aidé sur le vrai compost ?
from pathlib import Path

cands = []
for d in ['/content/runs', '/content/drive/MyDrive/compost/runs',
          '/content/drive/MyDrive/compost/backups']:
    cands += Path(d).glob('train_*/weights/best.pt')
FINETUNED = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]   # le plus récent = le fine-tuné (6d)
print('Modèle FINE-TUNÉ évalué sur le COMPOST :', FINETUNED)

!python scripts/evaluate.py --weights {FINETUNED} --data /content/captures_test/data.yaml --split test --runs-dir /content/runs

from IPython.display import Image, display
evals = sorted(Path('/content/runs').glob('eval_test_*'), key=lambda p: p.stat().st_mtime)
if evals:
    print('--- confusion COMPOST (APRÈS fine-tuning) ---')
    for img in sorted(evals[-1].rglob('*confusion*.png')):
        print(img.name); display(Image(str(img)))

In [ ]:
# 7. Copie du run complet (poids + métriques) vers Drive
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
!ls /content/drive/MyDrive/compost/runs